# V2 trajectory and early-warning analysis

Chronological early-warning evaluation for the V2 hierarchical geometry model on the untouched temporal view: per-robot episodes in operation order, fixed commissioning versus guarded short-term baselines (absolute displacement, velocity, trend, persistence/CUSUM, disagreement), a survival head fitted on allowed pre-cutoff data only, and censored one-day/seven-day risk metrics with review-sized outputs and figures.

Key features:
- Direct parameters: configure via `TrajectoryParams` (e.g. `params = TrajectoryParams(max_files_per_robot=64)`) or `V2_DATA_ROOT` / `V2_CHECKPOINT_PATH` / `V2_OUTPUT_ROOT` environment variables.
- Manual paths: run from the repository root and set `SRC_DIR`, `V2_DATA_ROOT`, and `V2_CHECKPOINT_PATH` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.
- Chronological robot episodes: the temporal view is grouped per robot in `(start_time, end_time, file_id)` order; trajectories never bridge maintenance (`EpisodeKind.MAINTENANCE` starts a new segment while the fixed commissioning baseline is preserved).
- Fixed/short-term trajectories: every file reports Mahalanobis absolute displacement, velocity, rolling trend, one-sided persistence/CUSUM, and fixed-versus-short-term disagreement. Abnormal or quarantined files are scored but never update either baseline (suspect guard).
- Allowed survival data only: the censored `CensoredSurvivalRisk` head fits exclusively on pre-cutoff non-test files with observed failure times; sealed static/temporal ids are asserted disjoint. A single-class horizon on tiny data is recorded as `uncalibrated` with exact cohort counts (risk figures skipped) instead of falling back to test data or fabricated calibration.
- Censored 1d/7d evaluation: per-horizon usable cohorts (early-censored files excluded), AUROC/AUPRC, Brier scores, calibration curve/ECE, event recall with lead time and persistence, false-warning runs per robot-month, and conformal healthy coverage.
- Review-sized bounded outputs: `trajectory_metrics.json`, `risk_metrics.json`, `provenance.json`, and at most three figures; bulk scores stay in memory.

In [ ]:
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import torch

os.environ.setdefault("PYTHONHASHSEED", "0")

# ---- Repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V2_REPO_ROOT to the checkout path.
V2_REPO_ROOT = os.environ.get("V2_REPO_ROOT", ".")
SRC_DIR = os.environ.get("V2_SRC_DIR", str(Path(V2_REPO_ROOT) / "src"))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / "representation").is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        "Run from the repository root or set V2_REPO_ROOT / V2_SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation.data import collate_variable_files
from representation.v2_checkpoint import load_v2_checkpoint
from representation.v2_inference import V2InferencePipeline, patch_regime_ids
from representation.v2_risk import (
    CensoredSurvivalRisk,
    expected_feature_width,
    trajectory_feature_matrix,
)
from representation.v2_trajectory import TrajectoryTracker
from synth.chronicle import load_chronological
from synth.config import PatchConfig
from synth.patchify import Patchifier
from synth.schema import EpisodeKind, SampleLabel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Hardware] Compute device:", device)


In [ ]:
# ---- Explicit data / checkpoint / output roots (edit these one-line values; used exactly) ----
V2_DATA_ROOT = os.environ.get("V2_DATA_ROOT", "data/generated/chronicle-client")
V2_CHECKPOINT_PATH = os.environ.get("V2_CHECKPOINT_PATH", "checkpoints/v2_geometry.pt")
V2_OUTPUT_ROOT = os.environ.get("V2_OUTPUT_ROOT", "outputs/v2_trajectory")

@dataclass
class TrajectoryParams:
    """V2 trajectory/early-warning configuration (works from a repository checkout)."""
    data_root: str = V2_DATA_ROOT
    checkpoint_path: str = V2_CHECKPOINT_PATH
    output_root: str = V2_OUTPUT_ROOT
    batch_size: int = int(os.environ.get("V2_BATCH_SIZE", "8"))
    max_files_per_robot: int | None = int(os.environ["V2_MAX_FILES_PER_ROBOT"]) if "V2_MAX_FILES_PER_ROBOT" in os.environ else None
    risk_seed: int = int(os.environ.get("V2_RISK_SEED", "0"))

params = TrajectoryParams()
configured_root = Path(params.data_root).expanduser()
DATA_ROOT = configured_root if configured_root.is_absolute() else Path.cwd() / configured_root
MANIFEST_PATH = DATA_ROOT / "manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Chronological manifest not found at {MANIFEST_PATH}. Set V2_DATA_ROOT to the materialized "
        "chronicle root (generate it with notebooks/generate_chronological_factory.ipynb).")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
configured_ckpt = Path(params.checkpoint_path).expanduser()
CHECKPOINT_PATH = configured_ckpt if configured_ckpt.is_absolute() else Path.cwd() / configured_ckpt
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"V2 checkpoint not found at {CHECKPOINT_PATH}. Set V2_CHECKPOINT_PATH to the trained "
        "V2 checkpoint (see notebooks/train_v2_geometry.ipynb). Trajectory analysis never runs without it.")
print(f"[Config] data={DATA_ROOT} checkpoint={CHECKPOINT_PATH} batch_size={params.batch_size} "
      f"max_files_per_robot={params.max_files_per_robot} risk_seed={params.risk_seed}")

# Restore the pipeline with hierarchical references exactly as saved.
pipeline = V2InferencePipeline.load(CHECKPOINT_PATH, device=str(device))
payload = load_v2_checkpoint(CHECKPOINT_PATH, pipeline.model, expected_config=pipeline.config)
if not isinstance(payload.get("geometry"), dict):
    raise RuntimeError(f"Checkpoint at {CHECKPOINT_PATH} carries no fitted geometry references.")
tracker_state = payload.get("tracker")
if not isinstance(tracker_state, dict):
    raise RuntimeError(f"Checkpoint at {CHECKPOINT_PATH} carries no commissioned trajectory baseline.")
if pipeline.calibrator is None:
    raise RuntimeError("Trajectory analysis requires the validation-calibrated confidence layer; "
                       "retrain with notebooks/train_v2_geometry.ipynb calibration.")
print(f"[Provenance] fallback_order={pipeline.config.fallback_order()} "
      f"calibrator=restored checkpoint_risk={'present' if pipeline.risk is not None else 'absent (fit in-notebook)'} "
      f"horizons={tuple(pipeline.config.risk_horizons_days)}")
# Distinct calibration provenances (Finding 3): the operating elevated
# threshold and the conformal fit cohort are read from their own checkpoint
# records and labeled separately — neither is claimed dev-val-only unless
# its own record proves it. No single shared record conflates them.
operating_record = payload.get("operating_threshold")
confidence_cohort_record = payload.get("confidence_calibrator_fit_cohort")
operating_cohort_label = operating_record.get("fit_cohort") if isinstance(operating_record, dict) else "absent"
confidence_fit_label = confidence_cohort_record.get("cohort") if isinstance(confidence_cohort_record, dict) else "absent"
operating_n = operating_record.get("n_samples") if isinstance(operating_record, dict) else None
confidence_n = confidence_cohort_record.get("n_samples") if isinstance(confidence_cohort_record, dict) else None
confidence_small = confidence_cohort_record.get("small_sample") if isinstance(confidence_cohort_record, dict) else None
print(f"[Calibration] elevated_threshold={pipeline.elevated_threshold:.4f} "
      f"source={pipeline.elevated_threshold_source} operating_cohort={operating_cohort_label} "
      f"operating_n={operating_n} confidence_fit_cohort={confidence_fit_label} "
      f"confidence_n={confidence_n} small_sample={confidence_small} "
      "(cohorts below the reference floor stay disclosed small-sample)")


In [ ]:
# Chronological robot episodes with fixed/short-term baseline trajectories.
samples, manifest = load_chronological(DATA_ROOT)
by_id = {s.file_id: s for s in samples}
for view in ("test_temporal", "dev_train", "dev_val"):
    if view not in manifest["splits"] or not manifest["splits"][view]:
        raise RuntimeError(f"Chronicle view {view!r} at {DATA_ROOT} is empty; regenerate the dataset.")
temporal_files = [by_id[i] for i in manifest["splits"]["test_temporal"]]
patchifier = Patchifier(PatchConfig(patch_size=pipeline.config.patch_size, stride=pipeline.config.stride))

def fresh_tracker() -> TrajectoryTracker:
    clone = TrajectoryTracker(pipeline.config.d_model)
    clone.load_state_dict(tracker_state)
    return clone

def robot_of(sample) -> str:
    return sample.operation.robot_id

robots = sorted({robot_of(s) for s in temporal_files})
print(f"[Episodes] temporal files={len(temporal_files)} robots={robots}")

# Suspect guard: abnormal or quarantined files are scored identically but update
# neither the fixed nor the short-term baseline (absorption prevention).
SUSPECT_RULE = "file_label is ABNORMAL or quarantined"
TRAJECTORY_KEYS = ("displacement", "velocity", "trend", "persistence", "disagreement")
file_rows: list[dict[str, object]] = []
n_maintenance_breaks = 0
trackers = {robot: fresh_tracker() for robot in robots}
for robot in robots:
    stream = sorted((s for s in temporal_files if robot_of(s) == robot),
                    key=lambda s: (s.operation.start_time, s.operation.end_time, s.file_id))
    starts = [s.operation.start_time for s in stream]
    if any(b < a for a, b in zip(starts, starts[1:])):
        raise RuntimeError(f"temporal stream for {robot} is not chronological")
    if params.max_files_per_robot is not None:
        stream = stream[-params.max_files_per_robot:]
    for sample in stream:
        maintenance_reset = sample.episode is not None and sample.episode.kind is EpisodeKind.MAINTENANCE
        n_maintenance_breaks += int(bool(maintenance_reset))
        suspect = (sample.file_label is SampleLabel.ABNORMAL) or (
            sample.split_provenance is not None and sample.split_provenance.is_quarantined)
        base = collate_variable_files([sample], patchifier)
        count = base["patches"].shape[1]
        regimes = patch_regime_ids([sample], base["starts"], count)
        out = pipeline.score_patches(base["patches"], base["patch_pad_mask"], base["patch_valid_mask"],
                                     base["robot_idx"], base["program_idx"], regimes,
                                     tracker=trackers[robot],
                                     suspect_flags=torch.tensor([bool(suspect)]),
                                     maintenance_resets=torch.tensor([bool(maintenance_reset)]))
        # Explicit energy identities (Finding 1): the acute file/trajectory
        # chain consumes the boundary-trained context energy; the
        # hierarchical population energy is an independent view, never fused.
        assert out["file"]["energy_source"] == "context_energy"
        assert out["file_population"]["energy_source"] == "population_energy"
        if not out["trajectory"]:
            raise RuntimeError("trajectory scoring requires the commissioned checkpoint baseline")
        traj = {k: float(out["trajectory"][k][0]) for k in TRAJECTORY_KEYS}
        if any(v != v for v in traj.values()):
            raise RuntimeError(f"non-finite trajectory features for {sample.file_id}")
        file_rows.append({"file_id": sample.file_id, "robot": robot,
                           "start_time": sample.operation.start_time, "end_time": sample.operation.end_time,
                           "suspect": bool(suspect), "maintenance_reset": bool(maintenance_reset),
                           "tail_energy": float(out["file"]["tail_energy"][0]),
                           "elevated_fraction": float(out["file"]["elevated_fraction"][0]),
                           "population_tail": float(out["file_population"]["tail_energy"][0]),
                           **traj})
print(f"[Trajectories] scored {len(file_rows)} files across {len(robots)} robots; "
      f"maintenance breaks={n_maintenance_breaks}; suspect rule: {SUSPECT_RULE}")
pop_tail = torch.tensor([float(r["population_tail"]) for r in file_rows])
print(f"[Energies] acute chain=context_energy (tail/elevated feed trajectory/confidence/risk); "
      f"population_tail median={float(pop_tail.median()):.4f} "
      f"p95={float(torch.quantile(pop_tail, 0.95)):.4f} max={float(pop_tail.max()):.4f} "
      "(independent view, never fused)")


In [ ]:
# Survival head fitted on allowed pre-cutoff data only (never the sealed test views).
cutoff = float(manifest["calendar"]["cutoff_time"])
sealed_ids = set(manifest["splits"]["test_static"]) | set(manifest["splits"]["test_temporal"])
fit_candidates = [s for s in samples
                  if s.file_id not in sealed_ids
                  and s.operation.start_time < cutoff
                  and s.future_targets is not None
                  and s.future_targets.time_to_next_failure is not None
                  and not s.future_targets.is_censored]
if any(s.file_id in sealed_ids for s in fit_candidates):
    raise RuntimeError("sealed test files reached the survival fit cohort")
if not fit_candidates:
    raise RuntimeError("allowed pre-cutoff survival cohort is empty; regenerate the dataset")
n_fit_quarantined = sum(1 for s in fit_candidates
                      if s.split_provenance is not None and s.split_provenance.is_quarantined)
print(f"[Survival-fit] allowed pre-cutoff cohort={len(fit_candidates)} "
      f"(quarantined precursors included: {n_fit_quarantined}; sealed views excluded: {len(sealed_ids)})")

# Chronological per-robot trajectory features for the fit cohort from the commissioned baseline.
fit_trackers: dict[str, TrajectoryTracker] = {}
fit_features_list, fit_days_list, fit_event_list = [], [], []
fit_by_robot: dict[str, list] = {}
for sample in fit_candidates:
    fit_by_robot.setdefault(robot_of(sample), []).append(sample)
for robot in sorted(fit_by_robot):
    stream = sorted(fit_by_robot[robot],
                    key=lambda s: (s.operation.start_time, s.operation.end_time, s.file_id))
    tracker = fresh_tracker()
    fit_trackers[robot] = tracker
    for sample in stream:
        maintenance_reset = sample.episode is not None and sample.episode.kind is EpisodeKind.MAINTENANCE
        suspect = (sample.file_label is SampleLabel.ABNORMAL) or (
            sample.split_provenance is not None and sample.split_provenance.is_quarantined)
        base = collate_variable_files([sample], patchifier)
        count = base["patches"].shape[1]
        regimes = patch_regime_ids([sample], base["starts"], count)
        out = pipeline.score_patches(base["patches"], base["patch_pad_mask"], base["patch_valid_mask"],
                                     base["robot_idx"], base["program_idx"], regimes,
                                     tracker=tracker,
                                     suspect_flags=torch.tensor([bool(suspect)]),
                                     maintenance_resets=torch.tensor([bool(maintenance_reset)]))
        assert out["file"]["energy_source"] == "context_energy"
        assert out["file_population"]["energy_source"] == "population_energy"
        row = trajectory_feature_matrix(out["trajectory"], out["file"]["tail_energy"],
                                        out["file"]["elevated_fraction"])
        fit_features_list.append(row)
        assert sample.future_targets is not None and sample.future_targets.time_to_next_failure is not None
        fit_days_list.append(sample.future_targets.time_to_next_failure / 86400.0)
        fit_event_list.append(not sample.future_targets.is_censored)
fit_features = torch.cat(fit_features_list)
fit_days = torch.tensor(fit_days_list, dtype=torch.float32)
fit_event = torch.tensor(fit_event_list, dtype=torch.bool)
width = expected_feature_width()
if fit_features.shape[1] != width:
    raise RuntimeError(f"risk feature width {fit_features.shape[1]} != canonical {width}")
risk = CensoredSurvivalRisk(width, horizons_days=tuple(pipeline.config.risk_horizons_days))
# Strict allowed-data fit with no test-data fallback. Tiny profiles may carry no
# observed 1d failures outside the sealed failure episodes; that structural gap is
# recorded as uncalibrated (with exact cohort counts below) instead of being
# patched with test data or fabricated calibration.
risk_fit_error: str | None = None
try:
    risk.fit(fit_features, fit_days, fit_event, seed=params.risk_seed)
    print(f"[Survival-fit] horizons_days={risk.horizons_days} fitted on {fit_features.shape[0]} allowed files "
          "(future targets never entered encoder inputs)")
except ValueError as exc:
    risk_fit_error = (f"survival fit on allowed pre-cutoff data failed ({exc}); "
                      "no test-data fallback was attempted")
    print(f"[Survival-fit] UNCALIBRATED: {risk_fit_error}")


In [ ]:
# Censored one-day/seven-day evaluation on the untouched temporal view.
from sklearn.metrics import average_precision_score, roc_auc_score

from representation.v2_risk import CensoredSurvivalRisk as _CSR

traj_batch = {k: torch.tensor([float(r[k]) for r in file_rows]) for k in TRAJECTORY_KEYS}
risk_features = trajectory_feature_matrix(traj_batch,
                                            torch.tensor([float(r["tail_energy"]) for r in file_rows]),
                                            torch.tensor([float(r["elevated_fraction"]) for r in file_rows]))
by_id = {s.file_id: s for s in samples}
days_all = torch.tensor([by_id[r["file_id"]].future_targets.time_to_next_failure / 86400.0
                         if by_id[r["file_id"]].future_targets is not None
                         and by_id[r["file_id"]].future_targets.time_to_next_failure is not None
                         else float("nan") for r in file_rows], dtype=torch.float32)
event_all = torch.tensor([by_id[r["file_id"]].future_targets is not None
                          and not by_id[r["file_id"]].future_targets.is_censored
                          for r in file_rows], dtype=torch.bool)

# Cohort composition is always reported (early-censored files carry no horizon information).
cohort_composition: dict[str, dict[str, int]] = {}
for horizon in tuple(pipeline.config.risk_horizons_days):
    included, labels = _CSR.horizon_cohort(days_all, event_all, horizon)
    cohort_composition[f"{horizon}d"] = {"n_usable": int(included.sum().item()),
                                        "n_events": int(labels[included].sum().item())}
print(f"[Cohorts] temporal usable/event composition: {cohort_composition} (NaN times excluded)")

proba = None
fit_proba = None
operating_threshold = float("nan")
brier = {"risk_1d": float("nan"), "risk_7d": float("nan")}
horizon_metrics: dict[str, dict[str, float]] = {}
ece_7d = float("nan")
calibration_bins: list[list[float]] = []
recall_1d, recall_7d, leads_1d, leads_7d, persist = [], [], [], [], []
false_runs = 0
robot_days_total = 0.0
false_per_30d = float("nan")

if risk_fit_error is None:
    proba = risk.predict_proba(risk_features)
    brier = risk.brier_score(risk_features, days_all, event_all)
    print(f"[Risk] Brier 1d={brier['risk_1d']:.4f} 7d={brier['risk_7d']:.4f} (censored cohorts only)")

    # Fixed operating point from allowed fit data: 95th percentile of risk_7d over fit negatives.
    fit_proba = risk.predict_proba(fit_features)
    _, fit_label_7d = _CSR.horizon_cohort(fit_days, fit_event, risk.horizons_days[1])
    fit_negatives = fit_proba["risk_7d"][~fit_label_7d]
    if fit_negatives.numel() == 0:
        raise RuntimeError("allowed fit cohort carries no 7d negatives for the operating point")
    operating_threshold = float(torch.quantile(fit_negatives, 0.95))
    print(f"[Calibration] operating threshold={operating_threshold:.4f} "
          "(95th percentile of allowed-fit 7d negatives; test labels never fit it)")

    for horizon, key in zip(risk.horizons_days, ("risk_1d", "risk_7d")):
        included, labels = _CSR.horizon_cohort(days_all, event_all, horizon)
        scores = proba[key][included]
        truth = labels[included].to(dtype=torch.float32)
        n_pos = int(truth.sum().item())
        if n_pos == 0 or n_pos == truth.numel():
            auroc = auprc = float("nan")
        else:
            auroc = float(roc_auc_score(truth.tolist(), scores.tolist()))
            auprc = float(average_precision_score(truth.tolist(), scores.tolist()))
        horizon_metrics[key] = {"horizon_days": float(horizon), "n_usable": int(included.sum().item()),
                                "n_events": n_pos, "auroc": auroc, "auprc": auprc,
                                "brier": float(brier[key])}
        print(f"[Horizon {horizon}d] usable={int(included.sum().item())} events={n_pos} "
              f"AUROC={auroc:.4f} AUPRC={auprc:.4f} Brier={float(brier[key]):.4f}")

    # Calibration curve and ECE on the 7d usable cohort (10 equal-width bins).
    mask_7d, labels_7d = _CSR.horizon_cohort(days_all, event_all, risk.horizons_days[1])
    scores_7d = proba["risk_7d"][mask_7d]
    truth_7d = labels_7d[mask_7d].to(dtype=torch.float32)
    ece_num, ece_den = 0.0, 0
    for b in range(10):
        sel = (scores_7d >= b / 10.0) & (scores_7d < (b + 1) / 10.0 if b < 9 else scores_7d <= 1.0)
        if int(sel.sum().item()) == 0:
            calibration_bins.append([b / 10.0 + 0.05, float("nan"), 0.0])
            continue
        acc = float(truth_7d[sel].mean().item())
        conf = float(scores_7d[sel].mean().item())
        calibration_bins.append([(b + 0.5) / 10.0, acc, float(sel.sum().item()) / float(sel.numel())])
        ece_num += abs(acc - conf) * int(sel.sum().item())
        ece_den += int(sel.sum().item())
    ece_7d = ece_num / max(1, ece_den)
    print(f"[Calibration] 7d ECE={ece_7d:.4f} over {ece_den} usable temporal files")

    # Event recall, lead time, and persistence per recorded failure episode.
    failures = [e for e in manifest["episodes"] if e["kind"] == "failure"]
    risk_1d_all = proba["risk_1d"].tolist()
    risk_7d_all = proba["risk_7d"].tolist()
    for failure in failures:
        robot, fail_t = failure["robot_id"], float(failure["start_time"])
        idx = [i for i, r in enumerate(file_rows) if r["robot"] == robot]
        for horizon_s, scores, recalls, leads in (
                (86400.0, risk_1d_all, recall_1d, leads_1d),
                (604800.0, risk_7d_all, recall_7d, leads_7d)):
            window = [i for i in idx if 0.0 <= fail_t - float(file_rows[i]["end_time"]) <= horizon_s]
            warned = [i for i in window if scores[i] >= operating_threshold]
            recalls.append(bool(warned))
            if warned:
                first = min(warned, key=lambda i: float(file_rows[i]["end_time"]))
                leads.append((fail_t - float(file_rows[first]["end_time"])) / 86400.0)
                persist.append(sum(1 for i in window if scores[i] >= operating_threshold) / len(window))
    print(f"[Events] failures={len(failures)} recall_1d={sum(recall_1d)}/{len(recall_1d)} "
          f"recall_7d={sum(recall_7d)}/{len(recall_7d)}")

    # False-warning runs per 30 robot-days (warned files with no failure within 7d).
    for robot in sorted({r["robot"] for r in file_rows}):
        rows = sorted((r for r in file_rows if r["robot"] == robot), key=lambda r: r["start_time"])
        robot_days_total += (max(r["end_time"] for r in rows) - min(r["start_time"] for r in rows)) / 86400.0
        robot_fails = [float(e["start_time"]) for e in failures if e["robot_id"] == robot]
        in_run = False
        for pos, r in enumerate(rows):
            i = file_rows.index(r)
            warned = risk_7d_all[i] >= operating_threshold
            has_future_failure = any(0.0 <= f - r["end_time"] <= 604800.0 for f in robot_fails)
            if warned and not has_future_failure and not in_run:
                false_runs += 1
                in_run = True
            elif not warned:
                in_run = False
    false_per_30d = false_runs / max(1e-6, robot_days_total / 30.0)
    print(f"[False-alerts] runs={false_runs} robot_days={robot_days_total:.1f} per_30d={false_per_30d:.3f}")
else:
    print("[Risk] head uncalibrated on allowed data; 1d/7d risk metrics stay null "
          "with the recorded reason (cohort composition above; no test fallback)")

# Conformal healthy coverage from the restored calibrator on allowed fit negatives.
# This needs only the validation-calibrated confidence layer, not the risk fit.
_, eval_label_7d = _CSR.horizon_cohort(fit_days, fit_event, tuple(pipeline.config.risk_horizons_days)[1])
fit_neg_mask = ~eval_label_7d
# Conformal coverage honesty (Finding 3): the restored calibrator was fitted
# on its own recorded cohort (confidence_fit_label); this evaluation scores
# allowed-fit negatives that may overlap it. The value is therefore an
# overlapping/in-sample diagnostic, never independent coverage.
conformal_coverage_cohort = ("allowed pre-cutoff 7d negatives "
                             "(may overlap the restored-calibrator fit cohort)")
if int(fit_neg_mask.sum().item()) == 0:
    coverage = float("nan")
    conformal_coverage_status = "null (no allowed fit negatives)"
    print("[Coverage] no allowed fit negatives; coverage recorded as null")
else:
    # fit_features[:, 0] is the displacement column: the exact signal the
    # healthy-tail calibrator was fitted on.
    coverage = pipeline.calibrator.coverage(fit_features[:, 0][fit_neg_mask], level=0.95)
    conformal_coverage_status = ("in-sample diagnostic (evaluation negatives may overlap "
                                 "the restored-calibrator fit cohort; not independent coverage)")
    print(f"[Coverage] conformal healthy coverage={coverage:.3f} on allowed-fit negatives "
          f"[calibrator fit cohort: {confidence_fit_label}; {conformal_coverage_status}]")


In [ ]:
# Review-sized metrics, provenance-separated outputs, and figures.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(params.output_root).expanduser()
OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else Path.cwd() / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
disp = torch.tensor([float(r["displacement"]) for r in file_rows])
vel = torch.tensor([float(r["velocity"]) for r in file_rows])
persist_all = torch.tensor([float(r["persistence"]) for r in file_rows])
pop_tail_all = torch.tensor([float(r["population_tail"]) for r in file_rows])
trajectory_metrics = {
    "n_files": len(file_rows), "n_robots": len({r["robot"] for r in file_rows}),
    "maintenance_breaks": n_maintenance_breaks, "suspect_rule": SUSPECT_RULE,
    "displacement": {"median": float(disp.median()), "p95": float(torch.quantile(disp, 0.95)),
                       "max": float(disp.max())},
    "velocity": {"median": float(vel.median()), "p95": float(torch.quantile(vel, 0.95)),
                 "max": float(vel.max())},
    "persistence_max": float(persist_all.max()),
    "reference": "fixed commissioning baseline (restored) vs guarded short-term baseline",
    "energy_chain": ("acute file/trajectory/confidence/risk on context_energy; "
                     "population_energy reported as an independent view, never fused"),
    "population_tail": {"median": float(pop_tail_all.median()),
                        "p95": float(torch.quantile(pop_tail_all, 0.95)),
                        "max": float(pop_tail_all.max())},
}
calibrated = risk_fit_error is None
risk_metrics = {
    "status": "calibrated" if calibrated else "uncalibrated",
    "reason": risk_fit_error,
    "horizons_days": list(tuple(pipeline.config.risk_horizons_days)),
    "n_fit_files": int(fit_features.shape[0]),
    "n_fit_quarantined": n_fit_quarantined,
    "fit_source": "pre-cutoff non-test files with observed failure times only",
    "cohort_composition": cohort_composition,
    "operating_threshold": operating_threshold,
    "operating_point_source": ("95th percentile of allowed-fit 7d negatives"
                               if calibrated else None),
    "horizons": horizon_metrics,
    "ece_7d": ece_7d,
    "calibration_bins": calibration_bins,
    "event_recall_1d": sum(recall_1d) / max(1, len(recall_1d)) if recall_1d else None,
    "event_recall_7d": sum(recall_7d) / max(1, len(recall_7d)) if recall_7d else None,
    "median_lead_time_1d_days": float(torch.tensor(leads_1d).median()) if leads_1d else None,
    "median_lead_time_7d_days": float(torch.tensor(leads_7d).median()) if leads_7d else None,
    "warning_persistence": sum(persist) / max(1, len(persist)) if persist else None,
    "false_warning_runs": false_runs,
    "false_warning_runs_per_30_robot_days": false_per_30d,
    "conformal_healthy_coverage": coverage,
    "conformal_coverage_cohort": conformal_coverage_cohort,
    "conformal_coverage_status": conformal_coverage_status,
    "confidence_calibrator_fit_cohort": (dict(confidence_cohort_record)
                                         if isinstance(confidence_cohort_record, dict) else None),
    "confidence_fit_cohort": confidence_fit_label,
    "operating_threshold_record": (dict(operating_record)
                                   if isinstance(operating_record, dict) else None),
    "operating_threshold_cohort": operating_cohort_label,
}
(OUTPUT_DIR / "trajectory_metrics.json").write_text(json.dumps(trajectory_metrics, indent=2, sort_keys=True), encoding="utf-8")
(OUTPUT_DIR / "risk_metrics.json").write_text(json.dumps(risk_metrics, indent=2, sort_keys=True, default=str), encoding="utf-8")
provenance = {"data_root": str(DATA_ROOT), "manifest_path": str(MANIFEST_PATH),
              "config_hash": manifest["config_hash"], "checkpoint": str(CHECKPOINT_PATH),
              "checkpoint_config": pipeline.config.to_dict(),
              "reference_source": "restored-checkpoint commissioned baseline (fixed) + guarded short-term",
              "risk_source": ("notebook-fit CensoredSurvivalRisk on pre-cutoff non-test cohort"
                              if calibrated else f"none ({risk_fit_error})"),
              "operating_threshold": (dict(operating_record)
                                      if isinstance(operating_record, dict) else None),
              "operating_threshold_cohort": operating_cohort_label,
              "confidence_calibrator_fit_cohort": (dict(confidence_cohort_record)
                                                  if isinstance(confidence_cohort_record, dict) else None),
              "confidence_fit_cohort": confidence_fit_label,
              "conformal_coverage_status": conformal_coverage_status,
              "energy_chain": ("acute file/trajectory/confidence/risk on context_energy; "
                               "population_energy reported as an independent view, never fused"),
              "sealed_views": ["test_static", "test_temporal"],
              "n_temporal_files": len(file_rows), "n_fit_files": int(fit_features.shape[0])}
(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, sort_keys=True), encoding="utf-8")

shown_robots = sorted({r["robot"] for r in file_rows})[:3]
fig, axes = plt.subplots(len(shown_robots), 1, figsize=(8, 2.5 * len(shown_robots)), squeeze=False)
for ax, robot in zip(axes[:, 0], shown_robots):
    rows = sorted((r for r in file_rows if r["robot"] == robot), key=lambda r: r["start_time"])
    times = [(r["end_time"] - rows[0]["start_time"]) / 86400.0 for r in rows]
    ax.plot(times, [r["displacement"] for r in rows], marker="o", markersize=3, label="displacement")
    ax.plot(times, [r["persistence"] for r in rows], marker="s", markersize=3, label="persistence")
    ax.set_title(f"{robot}: fixed-baseline trajectory (chronological files)")
    ax.set_xlabel("days since first temporal file")
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "trajectory_timelines.png", dpi=100)
plt.close(fig)
written = ["trajectory_metrics.json", "risk_metrics.json", "provenance.json", "trajectory_timelines.png"]

if calibrated:
    assert proba is not None and fit_proba is not None
    fig, ax = plt.subplots(figsize=(5, 4))
    xs = [b[0] for b in calibration_bins]
    ys = [b[1] for b in calibration_bins]
    ax.plot(xs, ys, marker="o", label="observed")
    ax.plot([0, 1], [0, 1], linestyle="--", label="ideal")
    ax.set_title("7d risk calibration curve (censored cohort)")
    ax.set_xlabel("mean predicted risk")
    ax.set_ylabel("observed failure rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "calibration_curve.png", dpi=100)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.hist(proba["risk_7d"].tolist(), bins=20, alpha=0.7, label="temporal risk_7d")
    ax.hist(fit_proba["risk_7d"].tolist(), bins=20, alpha=0.7, label="allowed-fit risk_7d")
    ax.axvline(operating_threshold, color="red", linestyle="--", label="operating point")
    ax.set_title("7d risk score distributions")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "risk_histograms.png", dpi=100)
    plt.close(fig)
    written += ["calibration_curve.png", "risk_histograms.png"]
else:
    print("[Output] risk figures skipped: head uncalibrated on allowed tiny data "
          "(trajectory timelines still written)")
print(f"[Output] wrote {', '.join(written)} to {OUTPUT_DIR} "
      "(bulk scores not persisted; schema: metrics + provenance + review-sized figures)")
